# Commands

[_Commands_](https://jupyterlab.readthedocs.io/en/stable/user/commands.html) are globally defined, asynchronous actions which allow plugins in Jupyter Front Ends to communicate. They also generally power user-facing actions, such as the launcher, toolbars, keyboard shortcuts, and the context and main menus. 

In [ ]:
%pip install -q ipylab

In [ ]:
from ipylab import JupyterFrontEnd
from ipywidgets import Output, HBox

app = JupyterFrontEnd()

## Command Registry

### List all commands

In [ ]:
sorted(app.commands.list_commands())[:5]

### Explore commands

Beyond their `id`, commands contain a variety of other information.

In [ ]:
from ipywidgets import VBox, Select
from IPython.display import JSON, Markdown

command_list = Select(
    description="command id", 
    options=sorted(app.commands.list_commands()),
    value="console:create",
    rows=1
)
command_info = Output()

def on_command_select(*_):
    if not command_list.value:
        return
    def _on_described(result, error):
        command_info.clear_output()
        with command_info:
            if result:
                display(JSON(result, expanded=True))
            if error:
                command_info.append_stderr(error)
            
    app.commands.describe(command_list.value, {}, _on_described)

command_list.observe(on_command_select)
on_command_select()
VBox([command_list, command_info], layout={"display": "flex"})

### Create a new console

In [ ]:
app.commands.execute('console:create', {
    'insertMode': 'split-right',
    'kernelPreference': {
        'shutdownOnClose': True,
    }
})

### Change the theme

In [ ]:
app.commands.execute('apputils:change-theme', { 'theme': 'JupyterLab Dark' })

### Create a new terminal

In [ ]:
app.commands.execute('terminal:create-new')

### Ready event

Some functionalities might require the `JupyterFrontEnd` widget to be ready on the frontend first.

This is for example the case when listing all the available commands, or retrieving the version with `app.version`.

The `on_ready` method can be used to register a callback that will be fired when the frontend is ready.

In [ ]:
from ipylab import JupyterFrontEnd
from ipywidgets import Output

app = JupyterFrontEnd()
out = Output()

def init():
    # show the first 5 commands
    cmds = app.commands.list_commands()[:5]
    out.append_stdout(cmds)

app.on_ready(init)
out

Or using `asyncio`:

In [ ]:
import asyncio

app = JupyterFrontEnd()

out = Output()

async def init():
    await app.ready()
    cmds = app.commands.list_commands()[:5]
    out.append_stdout(cmds)

asyncio.create_task(init())
out

## Custom Commands

### Add your own command

Let's create a nice plot with `bqlot` and generate some random data.

See https://github.com/bqplot/bqplot/blob/master/examples/Advanced%20Plotting/Animations.ipynb for more details.

In [ ]:
import numpy as np

from bqplot import LinearScale, Lines, Bars, Axis, Figure
from ipywidgets import IntSlider

In [ ]:
xs = LinearScale()
ys1 = LinearScale()
ys2 = LinearScale()

x = np.arange(20)
y = np.cumsum(np.random.randn(20))
y1 = np.random.rand(20)

line = Lines(x=x, y=y, scales={'x': xs, 'y': ys1}, colors=['magenta'], marker='square')
bar = Bars(x=x, y=y1, scales={'x': xs, 'y': ys2}, colorpadding=0.2, colors=['steelblue'])

xax = Axis(scale=xs, label='x', grid_lines='solid')
yax1 = Axis(scale=ys1, orientation='vertical', tick_format='0.1f', label='y', grid_lines='solid')
yax2 = Axis(scale=ys2, orientation='vertical', side='right', tick_format='0.0%', label='y1', grid_lines='none')

fig = Figure(marks=[bar, line], axes=[xax, yax1, yax2], animation_duration=1000, layout={"flex": "1"})
fig

We now define a function to update the data.

In [ ]:
def update_data(d0: int=20, d1: int=20):
    line.y = np.cumsum(np.random.randn(d0))
    bar.y = np.random.rand(d1)

In [ ]:
update_data()

This function will now be called when the JupyterLab command is executed: commands can also use custom [icons](./icons.ipynb) in place of `icon_class`.

Register it:

In [ ]:
app.commands.add_command('update_data', execute=update_data, label="Update Data", icon_class="jp-PythonIcon")

Execute it!

In [ ]:
app.commands.execute('update_data')

The slider should now be moving and taking random values.

Also the list of commands gets updated with the newly added command:

In [ ]:
assert 'update_data' in app.commands.list_commands()

That's great, but the command doesn't visually show up in the palette yet. So let's add it!

### Add the command to the palette

In [ ]:
from ipylab.commands import CommandPalette

In [ ]:
palette = CommandPalette()

In [ ]:
palette.add_item('update_data', 'Python Commands')

Open the command palette on the left side and the command should show now be visible.

### Remove a command

To remove a command that was previously added:

In [ ]:
app.commands.remove_command('update_data')

We can also describe the shape of the function as JSON schema:

## Command Arguments

Many commands provide a description of their arguments as JSON schema. Custom commands can also provide a [description](#Custom-command-argument-description).

### Run with validation

Similarly, for commands that have `describedBy` running `execute` with `validate=True` will first check the command arguments, and return an error _without_ running the command.  

Some commands also provide return data: while this is _supposed_ to be JSON, practically some commands return handles to things that can't be serialized to JSON, like pointers to DOM elements or recursive structures.

In [ ]:
validated = Output()
def on_execute(result: str, error: str):
    with validated:
        validated.append_stdout(result)
        validated.append_stderr(error)
app.commands.execute('console:create', {"isPalette": 1234}, handler=on_execute, validate=True)
validated

### Custom command argument description

[Custom commands](#Custom-commands) can also [describe](#Command-argument-description) their arguments as JSON schema.

In [ ]:
args_schema = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "d0": {"type": "number", "default": 20},
        "d1": {"type": "number", "default": 20},
    }
}

(Re-)reregister it, but with `described_by`:

In [ ]:
app.commands.add_command('update_data', execute=update_data, label="Update Data", icon_class="jp-PythonIcon", described_by={"args": args_schema})

Describe it:

In [ ]:
custom_described = Output()
def on_custom_describe(result: dict[str, dict[str, ...]], error):
    with custom_described:
        display({"application/json": result["args"]}, raw=True)
app.commands.describe('update_data', {}, on_custom_describe)
custom_described

In [ ]:
custom_validated = Output()
def on_custom_execute(result: str | None, error: str | None):
    custom_validated.clear_output()
    with validated:
        custom_validated.append_stdout(result)
        custom_validated.append_stderr(error)

In [ ]:
HBox([fig, custom_validated])

In [ ]:
app.commands.execute('update_data', {"d0": False}, handler=on_custom_execute, validate=True)

In [ ]:
app.commands.execute('update_data', {"d0": 20, "d1": 5}, handler=on_custom_execute, validate=True)